# Sentiment-Analyse der Video-Captions mit RoBERTa

Dieses Notebook analysiert die Caption-Texte der TikTok-Videos mit einem RoBERTa-basierten Sentiment-Modell.

Ziele:
- Sentiment-Labels und Wahrscheinlichkeiten pro Caption bestimmen
- KI- und Real-Videos deskriptiv und inferenzstatistisch vergleichen
- den Zusammenhang zwischen Caption-Sentiment und Engagement-Rate prüfen
- auswertbare CSV-Dateien für die weitere Ergebnisdarstellung exportieren


In [1]:
from pathlib import Path
from math import sqrt

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from scipy.stats import chi2_contingency, mannwhitneyu, spearmanr
from sklearn.preprocessing import StandardScaler
from tqdm.auto import tqdm
import torch
from transformers import AutoTokenizer, pipeline

sns.set_theme(style="whitegrid", context="talk")
plt.rcParams["figure.dpi"] = 130
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

COLOR_AI = "#4C78A8"
COLOR_REAL = "#F5855B"
PALETTE = {"KI": COLOR_AI, "Real": COLOR_REAL}


In [ ]:
BASE_DIR = Path.cwd().resolve().parents[1]
DATA_PATH = BASE_DIR / "data" / "01_raw" / "videos_metadata" / "01_AI_AND_REAL_TIKTOK_VIDEOS_DATASET_2025.csv"
OUTPUT_DIR = BASE_DIR / "comments" / "results"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "cardiffnlp/twitter-roberta-base-sentiment-latest"
BATCH_SIZE = 32
TEXT_COLUMN = "video_caption"

RESULTS_CSV = OUTPUT_DIR / "01_caption_sentiment_results.csv"
MISSING_ENGAGEMENT_EXCLUDED_CSV = OUTPUT_DIR / "01_caption_sentiment_excluded_missing_engagement.csv"

print(f"Datensatz: {DATA_PATH}")
print(f"Ausgabeordner: {OUTPUT_DIR}")
print(f"Modell: {MODEL_NAME}")


Datensatz: /Users/hannahernstberger/Documents/Master_/data/01_raw/videos_metadata/01_AI_AND_REAL_TIKTOK_VIDEOS_DATASET_2025.csv
Ausgabeordner: /Users/hannahernstberger/Documents/Master_/comments/results
Modell: cardiffnlp/twitter-roberta-base-sentiment-latest


In [ ]:
keep_cols = [
    "video_id",
    "influencer_type",
    "influencer",
    "video_engagement_rate",
    "video_like_count",
    "video_comment_count",
    "video_view_count",
    TEXT_COLUMN,
]

df = pd.read_csv(DATA_PATH, usecols=keep_cols).copy()
df["video_id"] = df["video_id"].astype("string").str.strip()
df[TEXT_COLUMN] = df[TEXT_COLUMN].fillna("").astype(str).str.replace(r"\s+", " ", regex=True).str.strip()
df = df[df[TEXT_COLUMN] != ""].copy()
excluded_missing_engagement_df = df[df["video_engagement_rate"].isna()].copy()
df = df[df["video_engagement_rate"].notna()].copy()
df["Typ"] = df["influencer_type"].map({"ai": "KI", "real": "Real"}).fillna(df["influencer_type"])
df["caption_length_chars"] = df[TEXT_COLUMN].str.len()
df["caption_length_words"] = df[TEXT_COLUMN].str.split().str.len()

print(f"Auswertbare Captions: {len(df):,}")
print(f"Ausgeschlossen wegen fehlender Engagement-Rate: {len(excluded_missing_engagement_df):,}")
display(excluded_missing_engagement_df.head(20))
display(df.head(3))


Auswertbare Captions: 1,029
Ausgeschlossen wegen fehlender Engagement-Rate: 0


,video_id,video_caption,video_like_count,video_comment_count,video_view_count,influencer_type,influencer,video_engagement_rate


,video_id,video_caption,video_like_count,video_comment_count,video_view_count,influencer_type,influencer,video_engagement_rate,Typ,caption_length_chars,caption_length_words
0,7516032103390203178,let’s GO!! #ai #fyp #viral #Vlog #googleveo3 #...,752,4,42500,ai,ai.kalai,0.020447,KI,54,8
1,7517830042206899469,i'm Kalaï and i was literally prompted to serv...,701,17,45100,ai,ai.kalai,0.020510,KI,139,19
2,7516243181650988334,oop 🫢🫢 #fyp #trending #googleveo3 #ai #viral #...,390,9,42500,ai,ai.kalai,0.011906,KI,73,11


In [ ]:
overview_df = (
    df.groupby("Typ")
    .agg(
        Videos=("video_id", "count"),
        Captionzeichen_M=("caption_length_chars", "mean"),
        Captionwoerter_M=("caption_length_words", "mean"),
        Engagement_M=("video_engagement_rate", "mean"),
    )
    .round(3)
    .reset_index()
)
display(overview_df)


,Typ,Videos,Captionzeichen_M,Captionwoerter_M,Engagement_M
0,KI,301,117.645,17.645,0.041
1,Real,728,127.040,17.837,0.048


In [ ]:
if torch.cuda.is_available():
    pipeline_device = 0
    device_label = "cuda"
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    pipeline_device = "mps"
    device_label = "mps"
else:
    pipeline_device = -1
    device_label = "cpu"

print(f"Verwendetes Device für Sentimentanalyse: {device_label}")

max_length = 512
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, model_max_length=max_length)
sentiment_pipe = pipeline(
    task="sentiment-analysis",
    model=MODEL_NAME,
    tokenizer=tokenizer,
    device=pipeline_device,
    truncation=True,
    max_length=max_length,
    top_k=None,
)

texts = df[TEXT_COLUMN].tolist()
all_results = []

for start in tqdm(range(0, len(texts), BATCH_SIZE), desc="RoBERTa-Sentiment"):
    batch = texts[start:start + BATCH_SIZE]
    batch_results = sentiment_pipe(batch, truncation=True, padding=True, max_length=max_length)
    all_results.extend(batch_results)

label_to_index = {"negative": -1.0, "neutral": 0.0, "positive": 1.0}
rows = []
for text, result in zip(texts, all_results):
    scores = {entry["label"].lower(): entry["score"] for entry in result}
    dominant = max(scores, key=scores.get)
    sentiment_index = (
        scores.get("positive", 0.0)
        - scores.get("negative", 0.0)
    )
    rows.append(
        {
            TEXT_COLUMN: text,
            "caption_sentiment_label": dominant,
            "caption_sentiment_negative": scores.get("negative", 0.0),
            "caption_sentiment_neutral": scores.get("neutral", 0.0),
            "caption_sentiment_positive": scores.get("positive", 0.0),
            "caption_sentiment_index": sentiment_index,
        }
    )

sentiment_df = pd.DataFrame(rows)
df = pd.concat([df.reset_index(drop=True), sentiment_df], axis=1)
df.head()



Verwendetes Device für Sentimentanalyse: mps


Some weights of the model checkpoint at cardiffnlp/twitter-roberta-base-sentiment-latest were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use mps


RoBERTa-Sentiment:   0%|          | 0/33 [00:00<?, ?it/s]

,video_id,video_caption,video_like_count,video_comment_count,video_view_count,influencer_type,influencer,video_engagement_rate,Typ,caption_length_chars,caption_length_words,video_caption,caption_sentiment_label,caption_sentiment_negative,caption_sentiment_neutral,caption_sentiment_positive,caption_sentiment_index
0,7516032103390203178,let’s GO!! #ai #fyp #viral #Vlog #googleveo3 #...,752,4,42500,ai,ai.kalai,0.020447,KI,54,8,let’s GO!! #ai #fyp #viral #Vlog #googleveo3 #...,positive,0.003976,0.207688,0.788336,0.784360
1,7517830042206899469,i'm Kalaï and i was literally prompted to serv...,701,17,45100,ai,ai.kalai,0.020510,KI,139,19,i'm Kalaï and i was literally prompted to serv...,positive,0.009904,0.455263,0.534833,0.524929
2,7516243181650988334,oop 🫢🫢 #fyp #trending #googleveo3 #ai #viral #...,390,9,42500,ai,ai.kalai,0.011906,KI,73,11,oop 🫢🫢 #fyp #trending #googleveo3 #ai #viral #...,neutral,0.022336,0.635164,0.342500,0.320164
3,7516395955562859819,xoxo #loveisland #viral #vlog #fyp #trending #...,1225,12,42200,ai,ai.kalai,0.034408,KI,60,8,xoxo #loveisland #viral #vlog #fyp #trending #...,neutral,0.011116,0.679467,0.309417,0.298301
4,7518222079955602743,PSA 🩷 #advice #bigsisteradvice #therapy #thera...,443,7,18000,ai,ai.kalai,0.027056,KI,141,16,PSA 🩷 #advice #bigsisteradvice #therapy #thera...,neutral,0.027738,0.769567,0.202694,0.174956


In [6]:
# Es wird nur die zentrale Ergebnisdatei gespeichert; Auswertungstabellen entstehen im Comparison-Notebook.
df.to_csv(RESULTS_CSV, index=False)
excluded_missing_engagement_df.to_csv(MISSING_ENGAGEMENT_EXCLUDED_CSV, index=False)
print(f"Gespeichert: {RESULTS_CSV}")


Gespeichert: /Users/hannahernstberger/Documents/Master_/comments/results/01_caption_sentiment_results.csv
